<a href="https://colab.research.google.com/github/Leticianunes13/assistente-voz-ai/blob/main/assistente_voz_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================
# 🎤 1. GRAVAÇÃO DE ÁUDIO
# ==============================
language = 'pt'

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

RECORD = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def record(sec=5):
  display(Javascript(RECORD))
  js_result = output.eval_js(f'record({sec * 1000})')
  audio = b64decode(js_result.split(',')[1])
  file_name = 'request_audio.wav'
  with open(file_name, 'wb') as f:
    f.write(audio)
  return f'/content/{file_name}'

print('🎤 Gravando...')
record_file = record(5)
display(Audio(record_file))


# ==============================
# 🧠 2. TRANSCRIÇÃO (WHISPER)
# ==============================
!pip install -q git+https://github.com/openai/whisper.git

import whisper

model = whisper.load_model("small")
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"].strip()

print("📝 Transcrição:")
print(transcription)


!pip install -q groq

import os
from groq import Groq
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")

if not api_key:
    raise ValueError("❌ API KEY do Groq não encontrada!")

client = Groq(api_key=api_key)

def chamar_groq(texto):
    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "user", "content": texto}
        ]
    )
    return response.choices[0].message.content

resposta_final = chamar_groq(transcription)

print("💬 Resposta:")
print(resposta_final)


# ==============================
# 🔊 4. TEXTO → VOZ
# ==============================
!pip install -q gTTS

from gtts import gTTS

response_audio = "/content/response_audio.mp3"

gtts = gTTS(text=resposta_final, lang=language)
gtts.save(response_audio)

display(Audio(response_audio, autoplay=True))